# Dirty Cafe Sales - Data Cleaning

**Dataset:** `dirty_cafe_sales.csv`  
**Goal:** Understand the data, fix quality issues, and prepare a clean version for analysis.

---

The first rule I follow on any project: *never touch the raw data until you understand it.*  
This notebook documents every cleaning decision - and the reason behind each one.

## 1. Problem Understanding

This is a café sales dataset with transaction-level records. Before doing any analysis, I need to answer a few basic questions:

- What does the data actually look like?
- Are the data types correct?
- Are there missing values - and how are they hidden?
- Are calculated fields like `Total Spent` actually consistent?

Real-world datasets are almost never clean out of the box. This one turned out to have missing values disguised as text (`'error'`, `'unknown'`, `'none'`), wrong data types on every numeric column, and dates stored as plain strings.

## 2. Import Libraries

Only `pandas` and `numpy` are needed here. Visualization comes later in the EDA notebook.

In [1]:
import pandas as pd
import numpy as np

## 3. Load the Dataset

Loading the raw file exactly as-is - no changes yet.

In [2]:
df = pd.read_csv('dirty_cafe_sales.csv')
print(f"Shape: {df.shape}")
df.head()

Shape: (10000, 8)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


## 4. Dataset Overview

First, I want to see column names, data types, and how many non-null values exist per column.

This immediately reveals two things:
- All columns come in as `object` (string) - even `Quantity` and `Price Per Unit`
- Several columns already show null counts, meaning some values are truly missing (not just disguised as text)

I'll hold off on any fixes until I've looked at the actual values.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [4]:
# Null count per column before any cleaning
print("Null counts before cleaning:")
print(df.isna().sum())

Null counts before cleaning:
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64


In [5]:
# Check unique values in text columns to spot hidden garbage
for col in ['Item', 'Payment Method', 'Location']:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(12))


--- Item ---
Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
NaN          333
ERROR        292
Name: count, dtype: int64

--- Payment Method ---
Payment Method
NaN               2579
Digital Wallet    2291
Credit Card       2273
Cash              2258
ERROR              306
UNKNOWN            293
Name: count, dtype: int64

--- Location ---
Location
NaN         3265
Takeaway    3022
In-store    3017
ERROR        358
UNKNOWN      338
Name: count, dtype: int64


### What I Found

After scanning the raw values:

| Problem | Affected Columns |
|---|---|
| Numeric columns stored as text | `Quantity`, `Price Per Unit`, `Total Spent` |
| Date stored as text | `Transaction Date` |
| Hidden missing values (`error`, `unknown`, `na`, etc.) | `Item`, `Payment Method`, `Location` |
| Real NaN missing values | All columns except `Transaction ID` |

None of this is unusual for real datasets. The plan: fix each column type one by one, standardize all missing values to `NaN`, and validate the calculated field.

## 5. Raw Data Preservation

Before touching anything, I keep a backup of the original data.  
For each column I modify, I create a `_raw` copy first.

This is non-negotiable - it lets me compare before/after and ensures nothing is permanently lost.

In [6]:
# Save a full backup of the original file
df_original = df.copy()
df_original.to_csv('dirty_cafe_sales_raw_backup.csv', index=False)
print("Raw backup saved.")

Raw backup saved.


---

## 6. Data Cleaning Process

Cleaning in this order:
1. Text columns - `Location`, `Item`, `Payment Method`
2. Numeric columns - `Quantity`, `Price Per Unit`, `Total Spent`
3. Date column - `Transaction Date`
4. Logical validation - does `Total Spent = Quantity × Price Per Unit`?

### 6.1 Cleaning `Location`

**What's wrong:** The column has real values (`'In-store'`, `'Takeaway'`), but also junk entries like `'error'`, `'unknown'`, `'none'`, `'na'`, and empty strings. These aren't real locations  they're bad data entry masquerading as values.

**Fix:**
- Backup the original column
- Normalize to lowercase and strip whitespace (so `'  Error '` and `'error'` match)
- Replace all garbage tokens with `NaN`
- Add a binary indicator column (1 = missing, 0 = present)

The indicator column is useful later  it lets me quantify how many values were affected without losing that information.

In [7]:
# Step 1: Backup
df['location_raw'] = df['Location'].copy()

# Step 2: Normalize for comparison
s = df['Location'].astype(str).str.strip().str.lower()

# Step 3: Define garbage tokens
bad = {'', 'error', 'unknown', 'na', 'n/a', 'none', 'null', 'blank'}

# Step 4: Mask rows where value is a garbage token
mask = s.isin(bad)

# Step 5: Replace with NaN
df.loc[mask, 'Location'] = np.nan

# Step 6: Add missing indicator
df['location_missing'] = df['Location'].isna().astype(int)

# Verify
print("Missing count after cleaning:", df['Location'].isna().sum())
print(df['Location'].value_counts(dropna=False))

Missing count after cleaning: 3961
Location
NaN         3961
Takeaway    3022
In-store    3017
Name: count, dtype: int64


### 6.2 Cleaning `Item`

**What's wrong:** Same issue as Location - placeholder text mixed in with real item names like `'Coffee'` and `'Juice'`.

Using the same pattern as above keeps the code consistent and easy to follow.

In [8]:
# Backup
df['Item_raw'] = df['Item'].copy()

# Normalize
a = df['Item'].astype(str).str.strip().str.lower()

# Mask and replace
mask_item = a.isin(bad)
df.loc[mask_item, 'Item'] = np.nan

# Add indicator
df['Item_missing'] = df['Item'].isna().astype(int)

# Verify
print("Missing count after cleaning:", df['Item'].isna().sum())
print(df['Item'].value_counts(dropna=False))

Missing count after cleaning: 969
Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
NaN          969
Name: count, dtype: int64


### 6.3 Cleaning `Payment Method`

**What's wrong:** Same placeholder problem. This column has the highest missing rate of the three text columns - almost a third of all rows don't have a payment method recorded.

This is important context for the EDA: any payment analysis will only represent about two-thirds of actual transactions.

In [9]:
# Backup
df['Payment Method_raw'] = df['Payment Method'].copy()

# Normalize
b = df['Payment Method'].astype(str).str.strip().str.lower()

# Mask and replace
mask_payment = b.isin(bad)
df.loc[mask_payment, 'Payment Method'] = np.nan

# Add indicator
df['Payment Method_missing'] = df['Payment Method'].isna().astype(int)

# Verify
print("Missing values:", df['Payment Method'].isna().sum())
print(df['Payment Method'].value_counts(dropna=False))

Missing values: 3178
Payment Method
NaN               3178
Digital Wallet    2291
Credit Card       2273
Cash              2258
Name: count, dtype: int64


### 6.4 Cleaning Numeric Columns

**What's wrong:** `Quantity`, `Price Per Unit`, and `Total Spent` came in as `object` (string). Pandas can't do math on strings, so I need to convert them.

**Fix:** `pd.to_numeric()` with `errors='coerce'`. The coerce option means anything that can't become a number - like `'error'` or `'unknown'` -automatically becomes `NaN`. Clean and simple.

I keep the original string versions as backup before converting.

In [10]:
# Backup raw string versions
df['Quantity_raw'] = df['Quantity'].copy()
df['price_per_unit_raw'] = df['Price Per Unit'].copy()
df['Total_spent_raw'] = df['Total Spent'].copy()

# Convert to numeric — non-numeric values become NaN automatically
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')
df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce')

# Confirm
print("Data types after conversion:")
print(df[['Quantity', 'Price Per Unit', 'Total Spent']].dtypes)
print("\nMissing values after conversion:")
print(df[['Quantity', 'Price Per Unit', 'Total Spent']].isna().sum())

Data types after conversion:
Quantity          float64
Price Per Unit    float64
Total Spent       float64
dtype: object

Missing values after conversion:
Quantity          479
Price Per Unit    533
Total Spent       502
dtype: int64


In [11]:
# Sanity check on value ranges — do these make sense for a café?
df[['Quantity', 'Price Per Unit', 'Total Spent']].describe()

,Quantity,Price Per Unit,Total Spent
count,9521.000000,9467.000000,9498.000000
mean,3.028463,2.949984,8.924352
std,1.419007,1.278450,6.009919
min,1.000000,1.000000,1.000000
25%,2.000000,2.000000,4.000000
50%,3.000000,3.000000,8.000000
75%,4.000000,4.000000,12.000000
max,5.000000,5.000000,25.000000


### 6.5 Cleaning `Transaction Date`

**What's wrong:** Dates were stored as plain text strings. Pandas treats them like any other text, so I can't filter by date range or extract month/year.

**Fix:** `pd.to_datetime()` with `errors='coerce'`. Valid date strings get converted; anything invalid becomes `NaT` (the datetime equivalent of `NaN`). No manual filtering needed.

In [12]:
# Backup original date strings
df['Transaction_Date_raw'] = df['Transaction Date'].copy()

# Convert to proper datetime
df['Transaction Date'] = pd.to_datetime(
    df['Transaction Date'],
    errors='coerce'  # invalid strings → NaT
)

print("Date column dtype:", df['Transaction Date'].dtype)
print("Missing (NaT) count:", df['Transaction Date'].isna().sum())

Date column dtype: datetime64[us]
Missing (NaT) count: 460


## 7. Logical Validation - Total Spent Check

**The rule:** `Total Spent = Quantity × Price Per Unit`

This should always hold in a sales dataset. If it doesn't for some rows, one of the three values is incorrect. Better to catch that now.

**Two things to check:**
1. Do existing rows where all three values are present actually add up correctly?
2. For rows where `Total Spent` is missing but the other two exist, can I safely fill it in?

In [13]:
# What Total Spent should be based on Quantity and Price Per Unit
expected_total = df['Quantity'] * df['Price Per Unit']

# Rows where all three values exist but the math doesn't match
inconsistent_mask = (
    df['Total Spent'].notna() &
    expected_total.notna() &
    (df['Total Spent'] != expected_total)
)

print(f"Rows with calculation mismatch: {inconsistent_mask.sum()}")
df.loc[inconsistent_mask, ['Quantity', 'Price Per Unit', 'Total Spent']].head()

Rows with calculation mismatch: 0


,Quantity,Price Per Unit,Total Spent


In [22]:
# Where Total Spent is missing but both other values exist  fill it in
recompute_mask = (
    df['Total Spent'].isna() &
    df['Quantity'].notna() &
    df['Price Per Unit'].notna()
)

df.loc[recompute_mask, 'Total Spent'] = expected_total

print(f"Total Spent filled in for {recompute_mask.sum()} rows")
print(f"Remaining missing Total Spent: {df['Total Spent'].isna().sum()}")

Total Spent filled in for 0 rows
Remaining missing Total Spent: 40


In [15]:
# Audit column — flags rows where the original math was inconsistent
df['Total_mismatch'] = inconsistent_mask.astype(int)
print("Mismatch rows flagged:", df['Total_mismatch'].sum())

Mismatch rows flagged: 0


## 8. Post-Cleaning Review

Before saving, I want a final look at the full state of the dataframe.

In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Transaction ID          10000 non-null  str           
 1   Item                    9031 non-null   str           
 2   Quantity                9521 non-null   float64       
 3   Price Per Unit          9467 non-null   float64       
 4   Total Spent             9960 non-null   float64       
 5   Payment Method          6822 non-null   str           
 6   Location                6039 non-null   str           
 7   Transaction Date        9540 non-null   datetime64[us]
 8   location_raw            6735 non-null   str           
 9   location_missing        10000 non-null  int64         
 10  Item_raw                9667 non-null   str           
 11  Item_missing            10000 non-null  int64         
 12  Payment Method_raw      7421 non-null   str           
 13

In [17]:
print("Null counts per column after cleaning:")
print(df.isna().sum())

Null counts per column after cleaning:
Transaction ID               0
Item                       969
Quantity                   479
Price Per Unit             533
Total Spent                 40
Payment Method            3178
Location                  3961
Transaction Date           460
location_raw              3265
location_missing             0
Item_raw                   333
Item_missing                 0
Payment Method_raw        2579
Payment Method_missing       0
Quantity_raw               138
price_per_unit_raw         179
Total_spent_raw            173
Transaction_Date_raw       159
Total_mismatch               0
dtype: int64


## 9. Saving the Datasets

Three output files, each for a different purpose:

| File | Purpose | Notes |
|---|---|---|
| `transactions_raw.csv` | Audit / reference | Original column values before any cleaning |
| `dirty_cafe_clean.csv` | EDA / full analysis | All rows kept, all cleaning applied |
| `cafe_sales_dashboard.csv` | Visualizations | Rows with missing key fields dropped |

The clean file keeps all rows because different questions need different subsets. For example: revenue analysis only needs `Total Spent` to be present. Payment analysis only needs `Payment Method`. Dropping rows globally at this stage would throw away data that's still useful for other calculations.

In [18]:
# Raw file — original values for audit trail
raw_cols = [
    'Transaction ID',
    'Item_raw',
    'Quantity_raw',
    'price_per_unit_raw',
    'Total_spent_raw',
    'Payment Method_raw',
    'location_raw',
    'Transaction_Date_raw'
]

df_raw = df[raw_cols].copy()
df_raw.to_csv('transactions_raw.csv', index=False)
print(f"Raw file saved — shape: {df_raw.shape}")

Raw file saved — shape: (10000, 8)


In [23]:
# Full cleaned file - all rows, used for EDA
df_clean = df.copy()
df_clean.to_csv('dirty_cafe_clean.csv', index=False)
print(f"Clean file saved — shape: {df_clean.shape}")

Clean file saved — shape: (10000, 19)


In [24]:
# Dashboard file - only business columns, drop rows where charts would break
dashboard_cols = [
    'Transaction ID', 'Item', 'Quantity',
    'Price Per Unit', 'Total Spent',
    'Payment Method', 'Location', 'Transaction Date'
]

df_dashboard = df[dashboard_cols].copy()
df_dashboard = df_dashboard.dropna(subset=['Item', 'Total Spent'])
df_dashboard = df_dashboard.dropna(subset=['Payment Method', 'Location', 'Transaction Date'])

print(f"Dashboard file — shape: {df_dashboard.shape}")
df_dashboard.info()

Dashboard file — shape: (3560, 8)
<class 'pandas.DataFrame'>
Index: 3560 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    3560 non-null   str           
 1   Item              3560 non-null   str           
 2   Quantity          3411 non-null   float64       
 3   Price Per Unit    3379 non-null   float64       
 4   Total Spent       3560 non-null   float64       
 5   Payment Method    3560 non-null   str           
 6   Location          3560 non-null   str           
 7   Transaction Date  3560 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(3), str(4)
memory usage: 250.3 KB


In [21]:
df_dashboard.to_csv('cafe_sales_dashboard.csv', index=False)
print("Dashboard file saved.")

Dashboard file saved.


## 10. Summary

Here's what this notebook accomplished:

| Step | What was fixed |
|---|---|
| Placeholder text → NaN | `Location`, `Item`, `Payment Method` |
| String → numeric | `Quantity`, `Price Per Unit`, `Total Spent` |
| String → datetime | `Transaction Date` |
| Validated calculation | `Total Spent = Quantity × Price Per Unit` |
| Backed up originals | All modified columns |
| Saved three output files | Raw, clean, dashboard-ready |

**Key principle:** No rows were dropped from the main clean file. Row dropping only happens at analysis time, and only for the specific columns a given calculation requires.

The EDA notebook picks up from `dirty_cafe_clean.csv`.